# Anchored PC: smaller-rate follow-up

Run the same frozen Stage A source and checkpoint with latent rates 0.001 and 0.0001. Keep T/2T at 30/60, 60/120 and 120/240, the residual limit at 0.001 and the parameter-gradient change limit at 1%. The driver verifies that only the rate grid and output directory changed, and compares the actual checkpoint parameters, batch arrays, masks and keys with the first run. It continues the remaining Stage A probes only if the convergence gate passes. No training loop runs.

This notebook is prepared but has not been executed. Use a T4 runtime. The three input files are the original `anchored_pc_bundle.tar.gz`, the saved `colab-results.zip`, and `low_rates_driver.py`.

In [ ]:
from pathlib import Path
from google.colab import files
needed = ['anchored_pc_bundle.tar.gz', 'colab-results.zip', 'low_rates_driver.py']
if any(not (Path('/content') / name).exists() for name in needed):
    uploaded = files.upload()
assert all((Path('/content') / name).exists() for name in needed), needed


In [ ]:
import hashlib, json, tarfile, zipfile, os, sys, subprocess
expected = {
    'anchored_pc_bundle.tar.gz': '65a1fdb42df0bace3c9c030f76435477229cd5dc7416a5648753f7a78d6a68c9',
    'colab-results.zip': '249601c40a25b5781580204990f6fe6a3ba9fa08aa7c7680e8b825d0e8ba638a',
    'low_rates_driver.py': '2c88e6a5876c83790420a538acbcd11f7392b242ddeccfe59a9d650552624b51',
}
for name, digest in expected.items():
    assert hashlib.sha256((Path('/content') / name).read_bytes()).hexdigest() == digest, name
root = Path('/content/anchored_pc')
root.mkdir(exist_ok=True)
with tarfile.open('/content/anchored_pc_bundle.tar.gz') as archive:
    archive.extractall(root, filter='data')
manifest = json.loads((root / 'bundle_manifest.json').read_text())
for name, digest in manifest['files'].items():
    assert hashlib.sha256((root / name).read_bytes()).hexdigest() == digest, name
previous = root / 'results_20261024T122712Z'
previous.mkdir(exist_ok=True)
with zipfile.ZipFile('/content/colab-results.zip') as archive:
    assert all(not Path(name).is_absolute() and '..' not in Path(name).parts for name in archive.namelist())
    archive.extractall(previous)
(root / 'src/tests/__init__.py').touch(exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'jax-cuda13-plugin', 'jax-cuda13-pjrt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'jax[cuda12]==0.11.2', 'optax==0.2.8', 'flax==0.12.10',
    'numpy==2.5.4', 'tokenizers==0.23.3', 'pytest', 'hypothesis',
    'matplotlib', 'datasets', 'optuna', 'tqdm', 'jaxtyping'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', str(root / 'src')], check=True)
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['PYTHONPATH'] = str(root / 'src')
os.chdir(root / 'src')
probe = subprocess.run([sys.executable, '-c', "import jax; assert jax.devices()[0].platform == 'gpu'; print(jax.__version__, jax.devices())"], capture_output=True, text=True, check=True)
print(probe.stdout, probe.stderr)
print('Frozen source, checkpoint, caches and prior evidence restored and verified')


In [ ]:
import datetime, threading, shutil
run_dir = root / ('low_rates_' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
log_path = root / 'low_rates_process.log'
command = [sys.executable, '-u', '/content/low_rates_driver.py', '--root', str(root), '--output', str(run_dir)]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
watchdog = threading.Timer(7200, process.kill)
watchdog.daemon = True
watchdog.start()
try:
    with log_path.open('w') as log:
        for line in process.stdout:
            log.write(line)
            log.flush()
            print(line, end='', flush=True)
    code = process.wait()
    print('Process exit code:', code)
finally:
    watchdog.cancel()
if run_dir.exists():
    shutil.copy(log_path, run_dir / 'process.log')
    shutil.copy('/content/low_rates_driver.py', run_dir / 'driver.py')
if (run_dir / 'summary.json').exists():
    result = json.loads((run_dir / 'summary.json').read_text())
    print(json.dumps({key: result.get(key) for key in ['status', 'decision', 'reason', 'elapsed_seconds', 'selected_solver']}, indent=2))
assert code == 0, 'The driver failed or reached the process timeout; inspect the saved evidence'


In [ ]:
evidence = shutil.make_archive(str(run_dir), 'zip', run_dir)
print('Evidence:', evidence)
print('SHA256:', hashlib.sha256(Path(evidence).read_bytes()).hexdigest())
files.download(evidence)
